In [ ]:
!pip install lightgbm
!pip install catboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV,RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from pandas.core.arrays import categorical
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBRegressor
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from google.colab import files
import zipfile
from google.colab import files
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from nltk import NaiveBayesClassifier
import random
from nltk.corpus import names

In [ ]:
upload=files.upload()
with zipfile.ZipFile("spaceship-titanic.zip","r")as zip_ref:
  zip_ref.extractall()
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")

Saving spaceship-titanic.zip to spaceship-titanic (1).zip


In [ ]:
print(train.shape)
print(train.info())
print(train.columns)
print(train.head())
train.isna().sum()

(8693, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB
None
Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'Sho

,0
PassengerId,0
HomePlanet,201
CryoSleep,217
Cabin,199
Destination,182
Age,179
VIP,203
RoomService,181
FoodCourt,183
ShoppingMall,208


In [ ]:
X_train=train[["PassengerId","CryoSleep","VIP","Age","RoomService","VRDeck","FoodCourt","ShoppingMall","Spa","Cabin","Name"]].copy()
Y_train=train["Transported"]
X_test=test[["PassengerId","CryoSleep","VIP","Age","RoomService","VRDeck","FoodCourt","ShoppingMall","Spa","Cabin","Name"]].copy()

In [ ]:
X_train["total_spent"]=(X_train["RoomService"]+ X_train["FoodCourt"]+ X_train["ShoppingMall"]+ X_train["Spa"] +X_train["VRDeck"])
X_train["total_spent"] =np.where(X_train["Age"]<= 14,0,X_train["total_spent"])
X_train["Small_Age"]=X_train["Age"]<= 14
X_train["Big_Age"]=X_train["Age"]>14
X_train["total_spent"]=np.where(X_train["CryoSleep"]==True,0,X_train["total_spent"])
X_train[['CabLetter', 'CabNumber', 'CabSide']] =X_train['Cabin'].str.split('/', expand=True)
X_train["financial_situation"]=X_train["total_spent"].apply(lambda x:"poor" if x <= 5000
       else("middle" if x >5000 and x<= 20000 else "rich"))
X_train[["Id","group_id"]]=X_train['PassengerId'].str.split("_", expand=True)
train_g =X_train[X_train["group_id"].duplicated()]["group_id"]
X_train["in_group"] = X_train["group_id"].apply(lambda x: x in train_g.values)
import nltk
from nltk.corpus import names
import random

# 1. تجهيز الداتا لتدريب الموديل (مرة واحدة فقط)
nltk.download('names')
labeled_names = ([(name, 'male') for name in names.words('male.txt')] +
                 [(name, 'female') for name in names.words('female.txt')])
random.shuffle(labeled_names)

# 2. استخراج الميزات (آخر حرف)
def gender_features(word):
    return {'last_letter': word[-1]}

featuresets = [(gender_features(n), gender) for (n, gender) in labeled_names]

# 3. تدريب الموديل (خارج اللوب)
classifier = nltk.NaiveBayesClassifier.train(featuresets)

# 4. استخراج الأسماء من الداتا بتاعتك
# هناخد الاسم الأول بس (قبل أول مسافة) ونحوله لنص
X_train_names = X_train["Name"].astype(str).apply(lambda x: x.split()[0])

# 5. التوقع (استخدام الموديل بسرعة)
names_gender = [classifier.classify(gender_features(name)) for name in X_train_names]

# 6. إضافة النتيجة للجدول
X_train["gender"] = names_gender
print("Done! Gender column added.")

LookupError
  PassengerId CryoSleep    VIP   Age  RoomService  VRDeck  FoodCourt  \
0     0001_01     False  False  39.0          0.0     0.0        0.0   
1     0002_01     False  False  24.0        109.0    44.0        9.0   
2     0003_01     False   True  58.0         43.0    49.0     3576.0   
3     0003_02     False  False  33.0          0.0   193.0     1283.0   
4     0004_01     False  False  16.0        303.0     2.0       70.0   

   ShoppingMall     Spa  Cabin  ... total_spent  Small_Age  Big_Age  \
0           0.0     0.0  B/0/P  ...         0.0      False     True   
1          25.0   549.0  F/0/S  ...       736.0      False     True   
2           0.0  6715.0  A/0/S  ...     10383.0      False     True   
3         371.0  3329.0  A/0/S  ...      5176.0      False     True   
4         151.0   565.0  F/1/S  ...      1091.0      False     True   

   CabLetter CabNumber CabSide financial_situation    Id group_id in_group  
0          B         0       P                poor 

In [ ]:
X_test["total_spent"]=(X_test["RoomService"]+X_test["FoodCourt"]+ X_test["ShoppingMall"]+ X_test["Spa"] + X_test ["VRDeck"])
X_test["total_spent"] =np.where(X_test["Age"]<= 14,0,X_test["total_spent"])
X_test["Small_Age"]=X_test["Age"]<= 14
X_test["Big_Age"]=X_test["Age"]>14
X_test["total_spent"]=np.where(X_test["CryoSleep"]==True,0,X_test["total_spent"])
X_test[['CabLetter', 'CabNumber', 'CabSide']] =X_test['Cabin'].str.split('/', expand=True)
X_test["financial_situation"]=X_test["total_spent"].apply(lambda x:"poor" if x <= 5000
       else("middle" if x >5000 and x<= 20000 else "rich"))
X_test[["Id","group_id"]]=X_test['PassengerId'].str.split("_", expand=True)
test_g =X_test[X_test["group_id"].duplicated()]["group_id"]
X_test["in_group"] = X_test["group_id"].apply(lambda x: x in train_g.values)
X_test_names=[]
for n in X_test["Name"]:
  a=str(n).split( )
  X_test_names.append(a[0])
try:
  def gender_features(word):
    return {'last_letter': word[-1]}
  labeled_names = [(name, "female") for name in  names.words("female.txt")]
  + [(name, "male") for name in  names.words("male.txt")]
  random.shuffle(labeled_names)
  featuresets = [(gender_features(n), gender) for (n, gender) in labeled_names]
  classifier =NaiveBayesClassifier.test(featuresets)
  names_gender = []
  for i in X_test_names:
      names_gender.append(classifier.classify(gender_features(i)))
  X_test["gender"] = names_gender
except LookupError:
  print("LookupError")
finally:
  print(X_test.head())

LookupError
  PassengerId CryoSleep    VIP   Age  RoomService  VRDeck  FoodCourt  \
0     0013_01      True  False  27.0          0.0     0.0        0.0   
1     0018_01     False  False  19.0          0.0     0.0        9.0   
2     0019_01      True  False  31.0          0.0     0.0        0.0   
3     0021_01     False  False  38.0          0.0   585.0     6652.0   
4     0023_01     False  False  20.0         10.0     0.0        0.0   

   ShoppingMall     Spa  Cabin  ... total_spent  Small_Age  Big_Age  \
0           0.0     0.0  G/3/S  ...         0.0      False     True   
1           0.0  2823.0  F/4/S  ...      2832.0      False     True   
2           0.0     0.0  C/0/S  ...         0.0      False     True   
3           0.0   181.0  C/1/S  ...      7418.0      False     True   
4         635.0     0.0  F/5/S  ...       645.0      False     True   

   CabLetter CabNumber CabSide financial_situation    Id group_id in_group  
0          G         3       S                poor 

In [ ]:
catogrical_columns=["CryoSleep","VIP","CabSide","Small_Age","Big_Age","CabLetter"]
numeric_columns=["PassengerId","Age","VRDeck","RoomService","FoodCourt","ShoppingMall","Spa","total_spent","CabNumber"]

In [ ]:
numeric_features=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="mean")),
    ("scalar",StandardScaler())])
catogrical_features=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessing=ColumnTransformer(transformers=[
    ("num",numeric_features,numeric_columns),
    ("cat",catogrical_features,catogrical_columns)])
models={
    "RF":RandomForestClassifier(),
    "GBR": GradientBoostingClassifier(),
    "XGB":XGBClassifier(),
    "CBC": CatBoostClassifier(),
    "LGBM": LGBMClassifier()
}
param_grid={
    "RF":{
        "parameter__n_estimators":[100,200,150],
        "parameter__max_depth":[3,5,7]},
    "GBR":{
        "parameter__n_estimators":[100,200,150],
        "parameter__max_depth":[6,10,8],
        "parameter__learning_rate":[0.05,0.1,0.01]},
    "XGB":{
        "parameter__n_estimators":[50,120,200],
        "parameter__max_depth":[4,6,9],
        "parameter__learning_rate":[0.5,0.1,0.001]},
    "CBC":{
        "parameter__depth":[5,10,3],
        "parameter__iterations":[100,250,200],
        "parameter__learning_rate":[0.05,0.1,0.01],
        "parameter__verbose":[0]},
    "LGBM":{
        "parameter__n_estimators": [300, 250, 200],
        "parameter__learning_rate": [0.01, 0.05, 0.1],
        "parameter__max_depth": [3,5,7],
        "parameter__num_leaves": [15,31,63],
        "parameter__subsample": [0.7,0.8,1],
        "parameter__verbose":[-1]}

}
best_models={}

In [ ]:
%%time
for name,model in models.items():
  pipeline=Pipeline(steps=[
      ("preprocessing",preprocessing),
      ("parameter",model)])
  random_search=RandomizedSearchCV(
      estimator=pipeline,cv=3,scoring="accuracy",param_distributions=param_grid[name],n_jobs=-1,n_iter=8)
  random_search.fit(X_train,Y_train)
  print(name, random_search.best_score_)
  best_models[name] = random_search.best_estimator_

RF 0.7336967738057321
GBR 0.689415345146161
XGB 0.756474118415257
CBC 0.7332384730592771
LGBM 0.7518742765474767
CPU times: user 13.8 s, sys: 553 ms, total: 14.3 s
Wall time: 4min 51s


In [ ]:
bs_model=best_models["XGB"]
lgb_model=best_models["LGBM"]
RF_model=best_models["RF"]
bs_model_steps = bs_model.named_steps["parameter"]
importances = bs_model_steps.feature_importances_
for col, imp in zip(X_train.columns, importances):
    print(col, imp)
lgb_model_steps = lgb_model.named_steps["parameter"]
importances = lgb_model_steps.feature_importances_
for col, imp in zip(X_train.columns, importances):
    print(col, imp)
RF_model_steps = RF_model.named_steps["parameter"]
importances = RF_model_steps.feature_importances_
for col, imp in zip(X_train.columns, importances):
    print(col, imp)
pred1=bs_model.predict(X_test)
pred2=lgb_model.predict(X_test)
pred4=RF_model.predict(X_test)
final=(pred1 + pred2+pred4)/3
final=(final>0.5)

PassengerId 0.0041545546
CryoSleep 0.003393629
VIP 0.07701065
Age 0.052977737
RoomService 0.011659614
VRDeck 0.008487943
FoodCourt 0.036394887
ShoppingMall 0.34108078
Spa 0.0
Cabin 0.38870335
Name 0.0
total_spent 0.0
Small_Age 0.0
Big_Age 0.010188889
CabLetter 0.0
CabNumber 0.0
CabSide 0.0
financial_situation 0.0
Id 0.0
group_id 0.0
in_group 0.0
PassengerId 490
CryoSleep 368
VIP 847
Age 358
RoomService 725
VRDeck 254
FoodCourt 775
ShoppingMall 345
Spa 505
Cabin 200
Name 0
total_spent 0
Small_Age 0
Big_Age 356
CabLetter 6
CabNumber 0
CabSide 0
financial_situation 0
Id 0
group_id 0
in_group 37
PassengerId 0.0019357285315115102
CryoSleep 0.014251971386740862
VIP 0.08656224174533177
Age 0.105388209320575
RoomService 0.04895320783372311
VRDeck 0.038728948778984564
FoodCourt 0.07014148228887042
ShoppingMall 0.21453248458212076
Spa 0.00381315562632408
Cabin 0.14258164852416003
Name 0.21236180032496665
total_spent 5.629466275580945e-05
Small_Age 5.4800709038373286e-05
Big_Age 0.002087575805111

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
submission = pd.DataFrame({
    "PassengerId": X_test["PassengerId"],
    "Transported": final })
submission.to_csv("submission.csv", index=False)
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>